# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR<sup>2</sup> dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library via its Croissant schema.

### Dataset Source

This dataset is published with a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), providing structured access for programmatic use.

In [ ]:
# Ensure `mlcroissant` and pandas are installed
!pip install mlcroissant pandas

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get and print a summary from the metadata object
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

List available record sets, fields, and their `@id`s for exploration.

In [ ]:
# Explore the file and record set structure of the dataset
print("Record Sets available in this dataset:")
record_sets = dataset.record_sets
for record_set in record_sets:
    print(f"  - Name: {record_set.name}")
    print(f"    @id: {record_set.id}")
    print("    Fields:")
    for field in record_set.fields:
        print(f"      - Name: {field.name}")
        print(f"        @id: {field.id}")
        print(f"        Data type: {field.data_type}")
    print()

## 3. Data Extraction

Load record data from a specific record set into a pandas DataFrame, using the record set and field `@id`s discovered above.

In [ ]:
# Gather all record set @ids to load
record_set_ids = [record_set.id for record_set in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
        print()
    else:
        print(f"Record set {record_set_id} returned no records.")

# For further analysis, pick the main tabular record set (if only one, use it)
if len(dataframes) == 0:
    raise ValueError("No tabular data found in record sets.")
# Select first available record set for further processing
main_record_set_id = list(dataframes.keys())[0]
main_df = dataframes[main_record_set_id]
print(f"Columns in main record set {main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, and grouping. For demonstration, we select a numeric field and a group/categorical field based on the columns present. **All references use the field's `@id`.**

In [ ]:
# Pick a numeric field @id for analysis
# Let's heuristically select a numeric field and a group field from the dataframe
numeric_field_id = None
group_field_id = None

# Attempt to guess numeric fields from dataframe dtypes
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try to cast columns to numeric
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
            numeric_field_id = col
            break
        except Exception:
            continue

# Pick a group (categorical) field
for col in main_df.columns:
    if col != numeric_field_id and main_df[col].dtype == object and main_df[col].nunique() < len(main_df) / 2:
        group_field_id = col
        break

print(f"Numeric field for analysis (by @id): {numeric_field_id}")
print(f"Categorical/group field (by @id): {group_field_id}")

# Proceed if numeric field available
if numeric_field_id:
    # Remove obviously invalid/missing data if any
    numeric_values = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = numeric_values.quantile(0.10)  # Use 10th percentile as demonstration
    filtered_df = main_df[numeric_values > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id]) - numeric_values.mean()) / numeric_values.std()
    print(f"Normalized {numeric_field_id} (first 5 rows):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by group field if available
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id} (first 5 groups):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field and its relationship to a categorical/group field. Adjust fields using their `@id` as above.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(6,4))
    pd.to_numeric(main_df[numeric_field_id], errors='coerce').hist(bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by group field
    if group_field_id:
        plt.figure(figsize=(8,5))
        main_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.show()

## 6. Conclusion

In this notebook, we have loaded a clinical dataset for second primary colorectal cancer survivors using the `mlcroissant` library, discovered its record sets and fields by their `@id`, performed simple EDA—filtering and normalization on a numeric field, and visualized its distribution.

For more complex analysis, you can further explore the fields and relationships provided by the schema, always referencing them by `@id`. See the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/mlcroissant/) for more advanced usage.